# Toy Stripe-Parity ViT Loss Landscape: Raw Weights vs BigVAE Latents

This notebook replaces the MNIST benchmark with a deliberately difficult small synthetic task while keeping both parameterizations:

1. **Raw weights**: direct filter-wise normalized perturbations of the trained tiny ViT weights.
2. **BigVAE latents**: optional perturbations in frozen BigVAE latent slots, decoded back into the tiny ViT's 2D linear weights.

Toy task:

- 16x16 grayscale images.
- Patch size 4, so the ViT receives 16 patch tokens.
- Each patch independently contains vertical or horizontal stripes.
- Per-patch random noise, contrast, brightness, and slight phase jitter make it nontrivial local pattern recognition.
- A fixed subset `S` of 8 patch positions defines the binary label by parity/XOR:
  `y = sum(bits[S]) mod 2`.

The raw-weight landscape is intentionally hostile: XOR over 8 local patch types creates a high-order interaction, the dataset is small, and good solutions tend to be sharp. The notebook also exposes flat-parameter helpers for evaluating arbitrary raw perturbations.

Both raw and latent sections render the existing 2D contour plot plus an interactive Plotly 3D surface that can be rotated in the notebook.


In [ ]:
from __future__ import annotations

import itertools
import math
import random
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

try:
    from torch.func import functional_call
except Exception:
    from torch.nn.utils.stateless import functional_call

import matplotlib.pyplot as plt
from IPython.display import display

try:
    import plotly.graph_objects as go
except Exception:
    go = None

try:
    from omegaconf import OmegaConf
    from experiments.train_big_vae import _build_model_cfg, _normalize_model_state_dict_keys
    from models.weight_quantile_vae import BigWeightVAE, build_weight_quantile_vae
except Exception as exc:
    OmegaConf = None
    BigWeightVAE = None
    build_weight_quantile_vae = None
    _build_model_cfg = None
    _normalize_model_state_dict_keys = None
    print('BigVAE imports unavailable; raw-weight landscape still works:', repr(exc))

try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

plt.rcParams['figure.dpi'] = 120


@dataclass
class Config:
    seed: int = 42
    device: str = 'cuda:0' if torch.cuda.is_available() else 'cpu'

    # Synthetic stripe-parity task.
    image_size: int = 16
    patch_size: int = 4
    parity_subset: tuple[int, ...] = (0, 2, 5, 7, 8, 10, 13, 15)
    train_samples: int = 1024
    val_samples: int = 1024
    batch_size: int = 128
    eval_batch_size: int = 256
    num_workers: int = 0
    noise_std: float = 0.12
    contrast_min: float = 0.65
    contrast_max: float = 1.35
    brightness_jitter: float = 0.25
    phase_jitter_pixels: float = 0.35

    # Tiny ViT.
    hidden_dim: int = 48
    num_heads: int = 4
    mlp_ratio: float = 2.0
    num_classes: int = 2

    # Training. More epochs are intentional: parity is hard for a tiny 1-block ViT.
    train_epochs: int = 1000
    train_lr: float = 3e-3
    train_weight_decay: float = 1e-4
    grad_clip_norm: float = 1.0

    # Landscape grid.
    num_directions: int = 4
    grid_points: int = 31
    rho_max: float = 1.0
    landscape_split: str = 'train'  # train | val
    landscape_max_batches: int = 4  # 0 means full selected landscape loader for every grid point.
    rho_values: tuple[float, ...] = (0.25, 0.5, 0.75, 1.0)
    tau_values: tuple[float, ...] = (0.01, 0.05, 0.1)

    # Optional BigVAE latent landscape. Leave empty to run raw-only.
    big_vae_checkpoint: str = '/home/coder/project/artifacts/training/checkpoints/weight_quantile_vae_gpu0_square/stage_1/latest.pt'
    big_vae_tile_T_patches: int = 16
    big_vae_tile_d_out: int = 8
    big_vae_latent_init: str = 'random'  # random | base
    latent_base_mode: str = 'encoder'  # encoder | fit | encoder_fit | random
    latent_fit_steps: int = 0
    latent_fit_lr: float = 1e-1
    latent_task_train_steps: int = 1000
    latent_task_lr: float = 1e-1
    latent_task_weight_decay: float = 0.0
    latent_task_grad_clip_norm: float = 1.0
    latent_task_log_every: int = 50
    encoder_context_batches: int = 4
    encoder_context_rows: int = 1024
    artifact_dir: str = './artifacts/loss_landscape_stripe_parity_vit'


cfg = Config()
BIG_VAE_CHECKPOINT = cfg.big_vae_checkpoint  # Set path here, or leave empty for raw-only.
device = torch.device(cfg.device)
artifact_dir = Path(cfg.artifact_dir)
artifact_dir.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(cfg.seed)
assert cfg.image_size % cfg.patch_size == 0
assert cfg.hidden_dim % cfg.num_heads == 0
assert len(cfg.parity_subset) == 8
assert len(set(cfg.parity_subset)) == len(cfg.parity_subset)
assert min(cfg.parity_subset) >= 0
assert max(cfg.parity_subset) < (cfg.image_size // cfg.patch_size) ** 2

print(cfg)
print('device =', device)


In [ ]:
def generate_stripe_parity_dataset(
    num_samples: int,
    cfg: Config,
    *,
    seed: int,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Generate images, labels, and underlying patch-type bits.

    bits[:, p] = 0 means vertical stripes in patch p.
    bits[:, p] = 1 means horizontal stripes in patch p.
    Patch positions are row-major over the 4x4 patch grid.
    """
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(seed))

    patch_size = int(cfg.patch_size)
    grid_size = int(cfg.image_size // cfg.patch_size)
    num_patches = grid_size * grid_size

    bits = torch.randint(0, 2, (num_samples, num_patches), generator=generator, dtype=torch.long)
    labels = (bits[:, list(cfg.parity_subset)].sum(dim=1) % 2).long()
    images = torch.empty(num_samples, 1, cfg.image_size, cfg.image_size, dtype=torch.float32)

    yy, xx = torch.meshgrid(
        torch.arange(patch_size, dtype=torch.float32),
        torch.arange(patch_size, dtype=torch.float32),
        indexing='ij',
    )
    xx = xx.unsqueeze(0)
    yy = yy.unsqueeze(0)

    for patch_idx in range(num_patches):
        row = patch_idx // grid_size
        col = patch_idx % grid_size
        y0 = row * patch_size
        x0 = col * patch_size

        phase = (torch.rand(num_samples, 1, 1, generator=generator) * 2.0 - 1.0) * float(cfg.phase_jitter_pixels)
        contrast = float(cfg.contrast_min) + (float(cfg.contrast_max) - float(cfg.contrast_min)) * torch.rand(
            num_samples,
            1,
            1,
            generator=generator,
        )
        brightness = (torch.rand(num_samples, 1, 1, generator=generator) * 2.0 - 1.0) * float(cfg.brightness_jitter)

        vertical = torch.cos(math.pi * (xx + phase))
        horizontal = torch.cos(math.pi * (yy + phase))
        is_horizontal = bits[:, patch_idx].view(num_samples, 1, 1).bool()
        pattern = torch.where(is_horizontal, horizontal, vertical)
        noise = float(cfg.noise_std) * torch.randn(num_samples, patch_size, patch_size, generator=generator)
        patch = contrast * pattern + brightness + noise

        images[:, 0, y0:y0 + patch_size, x0:x0 + patch_size] = patch

    return images, labels, bits


def make_toy_loaders(cfg: Config):
    train_images, train_labels, train_bits = generate_stripe_parity_dataset(cfg.train_samples, cfg, seed=cfg.seed + 1)
    val_images, val_labels, val_bits = generate_stripe_parity_dataset(cfg.val_samples, cfg, seed=cfg.seed + 2)

    train_mean = train_images.mean()
    train_std = train_images.std().clamp_min(1e-6)
    train_images = (train_images - train_mean) / train_std
    val_images = (val_images - train_mean) / train_std

    pin_memory = device.type == 'cuda'
    train_loader = DataLoader(
        TensorDataset(train_images, train_labels),
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=pin_memory,
    )
    val_loader = DataLoader(
        TensorDataset(val_images, val_labels),
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=pin_memory,
    )
    data_info = {
        'train_images': train_images,
        'train_labels': train_labels,
        'train_bits': train_bits,
        'val_images': val_images,
        'val_labels': val_labels,
        'val_bits': val_bits,
        'train_mean': train_mean,
        'train_std': train_std,
    }
    return train_loader, val_loader, data_info


def plot_examples(data_info: dict[str, torch.Tensor], cfg: Config, n: int = 8, save_path: Path | None = None) -> None:
    images = data_info['train_images'][:n]
    labels = data_info['train_labels'][:n]
    bits = data_info['train_bits'][:n]
    fig, axes = plt.subplots(1, n, figsize=(1.7 * n, 1.8))
    if n == 1:
        axes = [axes]
    for idx, ax in enumerate(axes):
        ax.imshow(images[idx, 0], cmap='gray')
        subset_bits = ''.join(str(int(bits[idx, p])) for p in cfg.parity_subset)
        ax.set_title('y={}\n{}'.format(int(labels[idx]), subset_bits), fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=180, bbox_inches='tight')
    plt.show()


train_loader, val_loader, data_info = make_toy_loaders(cfg)
print('train batches:', len(train_loader), 'val batches:', len(val_loader))
print('label balance train:', torch.bincount(data_info['train_labels'], minlength=2).tolist())
print('label balance val:', torch.bincount(data_info['val_labels'], minlength=2).tolist())
plot_examples(data_info, cfg, n=8, save_path=artifact_dir / 'stripe_parity_examples.png')


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, mlp_ratio: float) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(round(dim * mlp_ratio))
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm1(x)
        attn, _ = self.attn(h, h, h, need_weights=False)
        return self.mlp(self.norm2(attn))


class TinyStripeParityViT(nn.Module):
    def __init__(self, cfg: Config) -> None:
        super().__init__()
        self.cfg = cfg
        patch_dim = int(cfg.patch_size * cfg.patch_size)
        self.num_patches = int((cfg.image_size // cfg.patch_size) ** 2)
        self.patch_embed = nn.Linear(patch_dim, cfg.hidden_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, cfg.hidden_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, cfg.hidden_dim))
        self.block = TransformerBlock(cfg.hidden_dim, cfg.num_heads, cfg.mlp_ratio)
        self.norm = nn.LayerNorm(cfg.hidden_dim)
        self.head = nn.Linear(cfg.hidden_dim, cfg.num_classes)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.pos_embed, std=0.02)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def patchify(self, images: torch.Tensor) -> torch.Tensor:
        p = int(self.cfg.patch_size)
        b = int(images.shape[0])
        patches = images.unfold(2, p, p).unfold(3, p, p).contiguous()
        return patches.view(b, 1, -1, p, p).squeeze(1).flatten(2).contiguous()

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        b = int(images.shape[0])
        x = self.patch_embed(self.patchify(images))
        cls = self.cls_token.expand(b, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embed
        x = self.block(x)
        x = self.norm(x)
        return self.head(x[:, 0])


@torch.no_grad()
def evaluate_loss(
    model: nn.Module,
    loader: DataLoader,
    *,
    max_batches: int | None = None,
) -> tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    for batch_idx, (images, labels) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        loss = F.cross_entropy(logits, labels, reduction='sum')
        total_loss += float(loss.detach().cpu())
        total_correct += int((logits.argmax(dim=-1) == labels).sum().detach().cpu())
        total += int(labels.numel())
    return total_loss / max(1, total), total_correct / max(1, total)


def train_model(cfg: Config, train_loader: DataLoader, val_loader: DataLoader) -> TinyStripeParityViT:
    model = TinyStripeParityViT(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.train_lr, weight_decay=cfg.train_weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,
        T_max=max(1, int(cfg.train_epochs)),
        eta_min=float(cfg.train_lr) * 0.05,
    )

    for epoch in range(1, int(cfg.train_epochs) + 1):
        model.train()
        running_loss = 0.0
        seen = 0
        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(images), labels)
            loss.backward()
            if cfg.grad_clip_norm > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg.grad_clip_norm))
            opt.step()
            running_loss += float(loss.detach().cpu()) * int(labels.numel())
            seen += int(labels.numel())
        scheduler.step()

        should_log = epoch == 1 or epoch % 20 == 0 or epoch == cfg.train_epochs
        if should_log:
            train_loss = running_loss / max(1, seen)
            val_loss, val_acc = evaluate_loss(model, val_loader)
            train_eval_loss, train_acc = evaluate_loss(model, train_loader)
            lr = scheduler.get_last_lr()[0]
            print(
                f'epoch={epoch:04d} lr={lr:.2e} '
                f'train_loss={train_loss:.4f} train_eval_loss={train_eval_loss:.4f} '
                f'train_acc={train_acc:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
            )
    return model


base_model = train_model(cfg, train_loader, val_loader)
base_train_loss, base_train_acc = evaluate_loss(base_model, train_loader)
base_val_loss, base_val_acc = evaluate_loss(base_model, val_loader)
base_state = {k: v.detach().clone() for k, v in base_model.state_dict().items()}
print('base_train_loss', base_train_loss, 'base_train_acc', base_train_acc)
print('base_val_loss', base_val_loss, 'base_val_acc', base_val_acc)
print('num_parameters', sum(p.numel() for p in base_model.parameters()))


In [ ]:
@torch.no_grad()
def collect_linear_input_contexts(
    model: TinyStripeParityViT,
    loader: DataLoader,
    cfg: Config,
) -> dict[str, torch.Tensor]:
    """Collect real X matrices for each 2D linear weight decoded by BigVAE.

    Returned tensors follow the BigVAE convention X=[n,d_in] for W=[d_in,d_out].
    """
    model.eval()
    contexts: dict[str, list[torch.Tensor]] = {
        'patch_embed.weight': [],
        'block.attn.in_proj_weight': [],
        'block.attn.out_proj.weight': [],
        'block.mlp.0.weight': [],
        'block.mlp.2.weight': [],
        'head.weight': [],
    }

    for batch_idx, (images, _labels) in enumerate(loader):
        if int(cfg.encoder_context_batches) > 0 and batch_idx >= int(cfg.encoder_context_batches):
            break
        images = images.to(device, non_blocking=True)
        b = int(images.shape[0])

        patches = model.patchify(images)
        contexts['patch_embed.weight'].append(patches.reshape(-1, patches.shape[-1]).detach().cpu())

        x = model.patch_embed(patches)
        cls = model.cls_token.expand(b, -1, -1)
        x = torch.cat([cls, x], dim=1) + model.pos_embed

        h = model.block.norm1(x)
        contexts['block.attn.in_proj_weight'].append(h.reshape(-1, h.shape[-1]).detach().cpu())

        qkv = F.linear(h, model.block.attn.in_proj_weight, model.block.attn.in_proj_bias)
        head_dim = int(cfg.hidden_dim // cfg.num_heads)
        qkv = qkv.view(b, h.shape[1], 3, cfg.num_heads, head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(dim=0)
        attn_pre_out = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=False)
        attn_pre_out = attn_pre_out.transpose(1, 2).contiguous().view(b, h.shape[1], cfg.hidden_dim)
        contexts['block.attn.out_proj.weight'].append(attn_pre_out.reshape(-1, attn_pre_out.shape[-1]).detach().cpu())

        attn = F.linear(attn_pre_out, model.block.attn.out_proj.weight, model.block.attn.out_proj.bias)

        mlp_in = model.block.norm2(attn)
        contexts['block.mlp.0.weight'].append(mlp_in.reshape(-1, mlp_in.shape[-1]).detach().cpu())
        mlp_hidden = F.gelu(model.block.mlp[0](mlp_in))
        contexts['block.mlp.2.weight'].append(mlp_hidden.reshape(-1, mlp_hidden.shape[-1]).detach().cpu())

        x = model.block.mlp[2](mlp_hidden)
        x = model.norm(x)
        contexts['head.weight'].append(x[:, 0].detach().cpu())

    output: dict[str, torch.Tensor] = {}
    for name, parts in contexts.items():
        if not parts:
            continue
        tensor = torch.cat(parts, dim=0).to(device=device, dtype=torch.float32)
        if int(cfg.encoder_context_rows) > 0 and int(tensor.shape[0]) > int(cfg.encoder_context_rows):
            tensor = tensor[: int(cfg.encoder_context_rows)].contiguous()
        output[name] = tensor
    return output


encoder_contexts = collect_linear_input_contexts(base_model, train_loader, cfg)
print({name: tuple(x.shape) for name, x in encoder_contexts.items()})


## Flat parameter helpers

These helpers make the benchmark easy to reuse for arbitrary raw-weight perturbations. `get_flat_params` extracts the current trainable parameter vector. `set_flat_params_` writes a flat vector back into the model. `loss_at_flat_params` evaluates loss/accuracy at any flat vector and restores the model by default.


In [ ]:
def named_trainable_parameters(model: nn.Module) -> list[tuple[str, nn.Parameter]]:
    return [(name, param) for name, param in model.named_parameters() if param.requires_grad]


def get_flat_params(model: nn.Module) -> torch.Tensor:
    parts = [param.detach().reshape(-1) for _name, param in named_trainable_parameters(model)]
    if not parts:
        return torch.empty(0, device=device)
    return torch.cat(parts).detach().clone()


@torch.no_grad()
def set_flat_params_(model: nn.Module, flat: torch.Tensor) -> None:
    offset = 0
    for _name, param in named_trainable_parameters(model):
        count = int(param.numel())
        chunk = flat[offset:offset + count].view_as(param).to(device=param.device, dtype=param.dtype)
        param.copy_(chunk)
        offset += count
    if offset != int(flat.numel()):
        raise ValueError(f'flat vector has {flat.numel()} elements, but model uses {offset}')


@torch.no_grad()
def loss_at_flat_params(
    model: nn.Module,
    flat: torch.Tensor,
    loader: DataLoader,
    *,
    max_batches: int | None = None,
    restore: bool = True,
) -> tuple[float, float]:
    old_flat = get_flat_params(model) if restore else None
    set_flat_params_(model, flat)
    loss, acc = evaluate_loss(model, loader, max_batches=max_batches)
    if restore:
        assert old_flat is not None
        set_flat_params_(model, old_flat)
    return loss, acc


base_flat = get_flat_params(base_model)
print('flat parameter dim:', int(base_flat.numel()))
print('loss_at_flat_params(base_flat, train):', loss_at_flat_params(base_model, base_flat, train_loader))


## Raw-weight 2D loss landscape

Directions use filter-wise normalization for 2D+ weight tensors. For each weight matrix/filter bank, Gaussian noise is normalized row-wise so each row has the same norm as the corresponding row of the base weight tensor. Biases, LayerNorm parameters, `cls_token`, and `pos_embed` are kept fixed in these directions.


In [ ]:
def filterwise_normalized_flat_direction(
    model: nn.Module,
    *,
    generator: torch.Generator,
) -> torch.Tensor:
    parts: list[torch.Tensor] = []
    for name, param in named_trainable_parameters(model):
        value = param.detach()
        if value.ndim >= 2 and name.endswith('weight'):
            noise = torch.randn(value.shape, generator=generator, dtype=value.dtype, device='cpu').to(value.device)
            flat_value = value.reshape(value.shape[0], -1)
            flat_noise = noise.reshape(noise.shape[0], -1)
            value_norm = flat_value.norm(dim=1, keepdim=True).clamp_min(1e-12)
            noise_norm = flat_noise.norm(dim=1, keepdim=True).clamp_min(1e-12)
            direction = (flat_noise / noise_norm * value_norm).reshape_as(value)
        else:
            direction = torch.zeros_like(value)
        parts.append(direction.reshape(-1))
    return torch.cat(parts).detach().clone()


def make_raw_direction_pairs(
    model: nn.Module,
    num_pairs: int,
    *,
    seed: int,
) -> list[tuple[torch.Tensor, torch.Tensor]]:
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(seed))
    return [
        (
            filterwise_normalized_flat_direction(model, generator=generator),
            filterwise_normalized_flat_direction(model, generator=generator),
        )
        for _ in range(int(num_pairs))
    ]


@dataclass
class LandscapeResult:
    kind: str
    direction_idx: int
    alphas: np.ndarray
    betas: np.ndarray
    losses: np.ndarray
    base_loss: float


def evaluate_landscape(
    *,
    kind: str,
    direction_pairs: list[Any],
    base_loss: float,
    eval_fn,
    grid_points: int,
    rho_max: float,
) -> list[LandscapeResult]:
    alphas = np.linspace(-float(rho_max), float(rho_max), int(grid_points))
    betas = np.linspace(-float(rho_max), float(rho_max), int(grid_points))
    results = []
    for direction_idx, pair in enumerate(direction_pairs):
        losses = np.full((len(alphas), len(betas)), np.nan, dtype=np.float64)
        t0 = time.time()
        for i, alpha in enumerate(alphas):
            for j, beta in enumerate(betas):
                losses[i, j] = float(eval_fn(pair, float(alpha), float(beta)))
        result = LandscapeResult(
            kind=kind,
            direction_idx=direction_idx,
            alphas=alphas,
            betas=betas,
            losses=losses,
            base_loss=float(base_loss),
        )
        results.append(result)
        print(
            f'{kind} direction={direction_idx} done in {time.time() - t0:.1f}s '
            f'min_delta={np.nanmin(losses - base_loss):.4f} max_delta={np.nanmax(losses - base_loss):.4f}'
        )
    return results


def compute_landscape_metrics(
    results: list[LandscapeResult],
    rho_values: Iterable[float],
    tau_values: Iterable[float],
) -> pd.DataFrame:
    rows = []
    for result in results:
        alpha_grid, beta_grid = np.meshgrid(result.alphas, result.betas, indexing='ij')
        delta = result.losses - result.base_loss
        step_alpha = float(abs(result.alphas[1] - result.alphas[0])) if len(result.alphas) > 1 else 1.0
        step_beta = float(abs(result.betas[1] - result.betas[0])) if len(result.betas) > 1 else 1.0
        cell_area = step_alpha * step_beta
        radius2 = alpha_grid * alpha_grid + beta_grid * beta_grid
        for rho in rho_values:
            disk = radius2 <= float(rho) ** 2 + 1e-12
            disk_delta = delta[disk]
            row = {
                'kind': result.kind,
                'direction_idx': result.direction_idx,
                'rho': float(rho),
                'S_rho': float(np.nanmax(disk_delta)),
                'M_rho': float(np.nanmean(disk_delta)),
                'disk_grid_points': int(disk.sum()),
            }
            for tau in tau_values:
                good = disk & (result.losses <= result.base_loss + float(tau))
                row[f'A_tau={tau:g}_rho'] = float(good.sum() * cell_area)
                row[f'A_frac_tau={tau:g}_rho'] = float(good.sum() / max(1, disk.sum()))
            rows.append(row)
    return pd.DataFrame(rows)


def plot_landscape(result: LandscapeResult, title: str | None = None, save_path: Path | None = None) -> None:
    alpha_grid, beta_grid = np.meshgrid(result.alphas, result.betas, indexing='ij')
    delta = result.losses - result.base_loss
    plt.figure(figsize=(5, 4))
    contour = plt.contourf(alpha_grid, beta_grid, delta, levels=40, cmap='magma')
    plt.colorbar(contour, label='L(alpha,beta) - L(0,0)')
    plt.contour(alpha_grid, beta_grid, delta, colors='black', linewidths=0.35, levels=12, alpha=0.55)
    plt.scatter([0], [0], c='cyan', s=20, edgecolors='black', linewidths=0.4)
    plt.xlabel('alpha')
    plt.ylabel('beta')
    plt.title(title or f'{result.kind} direction {result.direction_idx}')
    plt.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=180, bbox_inches='tight')
    plt.show()


def plot_landscape_3d(
    result: LandscapeResult,
    title: str | None = None,
    *,
    z_mode: str = 'delta',
    save_html_path: Path | None = None,
):
    """Interactive Plotly 3D surface for rotating the loss landscape in a notebook."""
    if go is None:
        print('Skipping interactive 3D plot: plotly is not installed. Install it with: pip install plotly')
        return None
    if z_mode not in {'delta', 'loss'}:
        raise ValueError(f'z_mode must be delta or loss, got {z_mode!r}')
    alpha_grid, beta_grid = np.meshgrid(result.alphas, result.betas, indexing='ij')
    z = result.losses - result.base_loss if z_mode == 'delta' else result.losses
    z_label = 'L(alpha,beta) - L(0,0)' if z_mode == 'delta' else 'L(alpha,beta)'
    base_z = 0.0 if z_mode == 'delta' else float(result.base_loss)
    fig = go.Figure(
        data=[
            go.Surface(
                x=alpha_grid,
                y=beta_grid,
                z=z,
                surfacecolor=z,
                colorscale='Magma',
                colorbar={'title': z_label},
                contours={
                    'z': {
                        'show': True,
                        'usecolormap': True,
                        'highlightcolor': 'white',
                        'project_z': True,
                    }
                },
                hovertemplate='alpha=%{x:.3f}<br>beta=%{y:.3f}<br>' + z_label + '=%{z:.5f}<extra></extra>',
            ),
            go.Scatter3d(
                x=[0.0],
                y=[0.0],
                z=[base_z],
                mode='markers',
                marker={'size': 5, 'color': 'cyan', 'line': {'color': 'black', 'width': 2}},
                name='base point',
            ),
        ]
    )
    fig.update_layout(
        title=title or f'{result.kind} direction {result.direction_idx}',
        scene={
            'xaxis_title': 'alpha',
            'yaxis_title': 'beta',
            'zaxis_title': z_label,
            'camera': {'eye': {'x': 1.55, 'y': -1.55, 'z': 1.1}},
        },
        width=850,
        height=650,
        margin={'l': 0, 'r': 0, 'b': 0, 't': 45},
    )
    if save_html_path is not None:
        save_html_path.parent.mkdir(parents=True, exist_ok=True)
        fig.write_html(save_html_path, include_plotlyjs='cdn')
    fig.show()
    return fig


def plot_metric_boxplots(metrics: pd.DataFrame, metric: str, save_path: Path | None = None) -> None:
    plt.figure(figsize=(6, 4))
    labels = []
    values = []
    for (kind, rho), group in metrics.groupby(['kind', 'rho']):
        labels.append(f'{kind}\nrho={rho:g}')
        values.append(group[metric].to_numpy())
    plt.boxplot(values, labels=labels, showmeans=True)
    plt.ylabel(metric)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=180, bbox_inches='tight')
    plt.show()


raw_model = TinyStripeParityViT(cfg).to(device)
raw_model.load_state_dict(base_state, strict=True)
raw_base_flat = get_flat_params(raw_model)
landscape_split = str(cfg.landscape_split).strip().lower()
if landscape_split not in {'train', 'val'}:
    raise ValueError(f'cfg.landscape_split must be train or val, got {cfg.landscape_split!r}')
landscape_loader = train_loader if landscape_split == 'train' else val_loader
landscape_max_batches = None if int(cfg.landscape_max_batches) <= 0 else int(cfg.landscape_max_batches)
print('landscape_split', landscape_split)
raw_base_loss, raw_base_acc = evaluate_loss(raw_model, landscape_loader, max_batches=landscape_max_batches)
print(f'raw_base_{landscape_split}_loss', raw_base_loss, f'raw_base_{landscape_split}_acc', raw_base_acc)
raw_direction_pairs = make_raw_direction_pairs(raw_model, cfg.num_directions, seed=cfg.seed + 1000)


def raw_eval_fn(pair, alpha: float, beta: float) -> float:
    d1, d2 = pair
    flat = raw_base_flat + float(alpha) * d1 + float(beta) * d2
    loss, _acc = loss_at_flat_params(raw_model, flat, landscape_loader, max_batches=landscape_max_batches, restore=False)
    return loss


raw_results = evaluate_landscape(
    kind='raw_fn',
    direction_pairs=raw_direction_pairs,
    base_loss=raw_base_loss,
    eval_fn=raw_eval_fn,
    grid_points=cfg.grid_points,
    rho_max=cfg.rho_max,
)
set_flat_params_(raw_model, raw_base_flat)
raw_metrics = compute_landscape_metrics(raw_results, cfg.rho_values, cfg.tau_values)
display(raw_metrics)
plot_landscape(
    raw_results[0],
    'Stripe-parity raw-weight FN landscape: direction 0',
    save_path=artifact_dir / 'raw_fn_direction_0_2d.png',
)
plot_landscape_3d(
    raw_results[0],
    'Interactive 3D raw-weight FN landscape: direction 0',
    save_html_path=artifact_dir / 'raw_fn_direction_0_3d.html',
)


## BigVAE latent-induced landscape

This section is optional. It loads a frozen BigVAE decoder and builds a latent parameterization of the tiny ViT's 2D linear weights. The initial latent point is produced by the BigVAE encoder from trained weights and real layer input activations `X`. Optional decoded-weight MSE refinement is available via `latent_base_mode='encoder_fit'` or `latent_base_mode='fit'`. Then the latent slots themselves are optimized on the stripe-parity cross-entropy task loss while the BigVAE decoder stays frozen.

Then random latent directions are sampled with tile-wise latent norm matching, the latent slots are perturbed by `(alpha, beta)`, decoded into raw tiny-ViT weights, and the same stripe-parity validation loss is evaluated.

The decoder sees fixed stage-like tiles: `d_in = patch_size_bigvae * big_vae_tile_T_patches`, `d_out = big_vae_tile_d_out`. Small matrices are packed by patch rows into these tiles to avoid paying one latent for mostly empty padding.


In [ ]:
@dataclass
class MatrixSpec:
    original_shape: tuple[int, ...]
    matrix_shape: tuple[int, int]
    transposed: bool


@dataclass
class TileSegment:
    name: str
    tile_row_start: int
    row_start: int
    row_len: int
    col_start: int
    col_len: int


@dataclass
class DecodeTile:
    key: str
    segments: list[TileSegment]


def tensor_to_matrix(tensor: torch.Tensor, spec: MatrixSpec) -> torch.Tensor:
    matrix = tensor.reshape(int(spec.original_shape[0]), -1)
    if spec.transposed:
        matrix = matrix.transpose(0, 1)
    return matrix.contiguous()


def matrix_to_tensor(matrix: torch.Tensor, spec: MatrixSpec) -> torch.Tensor:
    restored = matrix.transpose(0, 1) if spec.transposed else matrix
    return restored.contiguous().reshape(spec.original_shape)


def load_frozen_big_vae(checkpoint_path: str, device: torch.device):
    if not checkpoint_path:
        return None
    if BigWeightVAE is None:
        raise RuntimeError('BigVAE imports are unavailable in this kernel')
    path = Path(checkpoint_path).expanduser()
    payload = torch.load(path, map_location='cpu', weights_only=False)
    cfg_payload = OmegaConf.create(payload['config'])
    model_cfg = _build_model_cfg(cfg_payload)
    model = build_weight_quantile_vae(model_cfg)
    state = payload.get('model_state', payload.get('state_dict'))
    missing, unexpected = model.load_state_dict(_normalize_model_state_dict_keys(state), strict=False)
    if missing or unexpected:
        raise RuntimeError(f'BigVAE state mismatch: missing={missing[:8]} unexpected={unexpected[:8]}')
    model.to(device).eval()
    for param in model.parameters():
        param.requires_grad_(False)
    return model


class BigVAELatentAdapter(nn.Module):
    def __init__(self, big_vae: nn.Module, base_state: dict[str, torch.Tensor], cfg: Config) -> None:
        super().__init__()
        self.big_vae = big_vae
        self.patch_size = int(big_vae.cfg.patch_size)
        self.tile_T = int(cfg.big_vae_tile_T_patches)
        self.tile_d_in = self.patch_size * self.tile_T
        self.tile_d_out = int(cfg.big_vae_tile_d_out)
        self.use_dist = bool(getattr(big_vae, 'use_distribution_encoder', False))
        self.d_dist = int(big_vae.cfg.distribution.d_dist)
        self.specs: dict[str, MatrixSpec] = {}
        self.tiles: list[DecodeTile] = []
        self.direct_state = {k: v.detach().clone().to(device) for k, v in base_state.items()}
        self.latents = nn.ParameterDict()
        base_latent = big_vae.latent_base.detach().clone().to(device)

        for name, value in base_state.items():
            if not (value.ndim == 2 and name.endswith('weight')):
                continue
            spec = MatrixSpec(tuple(value.shape), (int(value.shape[1]), int(value.shape[0])), True)
            self.specs[name] = spec
            rows, cols = spec.matrix_shape
            pending: list[TileSegment] = []
            current_rows = 0

            def flush() -> None:
                nonlocal pending, current_rows
                if not pending:
                    return
                key = f't{len(self.tiles):05d}'
                self.tiles.append(DecodeTile(key=key, segments=list(pending)))
                if cfg.big_vae_latent_init == 'base':
                    latent = base_latent.clone()
                else:
                    latent = torch.randn_like(base_latent) * 0.02
                self.latents[key] = nn.Parameter(latent)
                pending = []
                current_rows = 0

            for col_start in range(0, cols, self.tile_d_out):
                col_len = min(self.tile_d_out, cols - col_start)
                for row_start in range(0, rows, self.patch_size):
                    row_len = min(self.patch_size, rows - row_start)
                    if current_rows > 0 and current_rows + row_len > self.tile_d_in:
                        flush()
                    pending.append(TileSegment(name, current_rows, row_start, row_len, col_start, col_len))
                    current_rows += row_len
                    if current_rows >= self.tile_d_in:
                        flush()
            flush()

    def latent_numel(self) -> int:
        return sum(param.numel() for param in self.latents.values())

    def decoded_numel(self) -> int:
        return sum(int(np.prod(self.direct_state[name].shape)) for name in self.specs)

    def decode_matrices(self, latent_override: dict[str, torch.Tensor] | None = None) -> dict[str, torch.Tensor]:
        result = {name: torch.zeros(spec.matrix_shape, device=device) for name, spec in self.specs.items()}
        if not self.tiles:
            return result
        keys = [tile.key for tile in self.tiles]
        latents = torch.stack([(latent_override or self.latents)[key] for key in keys], dim=0)
        batch = int(latents.shape[0])
        patch_mask = torch.zeros(batch, self.tile_T, device=device, dtype=torch.bool)
        d_in_mask = torch.zeros(batch, self.tile_d_in, device=device, dtype=torch.bool)
        d_out_mask = torch.zeros(batch, self.tile_d_out, device=device, dtype=torch.bool)
        for idx, tile in enumerate(self.tiles):
            used_rows = 0
            used_cols = 0
            for seg in tile.segments:
                row_end = seg.tile_row_start + seg.row_len
                d_in_mask[idx, seg.tile_row_start:row_end] = True
                used_rows = max(used_rows, row_end)
                used_cols = max(used_cols, seg.col_len)
            patch_mask[idx, :math.ceil(used_rows / self.patch_size)] = True
            d_out_mask[idx, :used_cols] = True
        dist_patch = torch.zeros(batch, self.tile_T, self.d_dist, device=device, dtype=latents.dtype) if self.use_dist else None
        decoded = self.big_vae._decode_from_latent_slots(
            latents,
            dist_patch_by_patch=dist_patch,
            patch_mask=patch_mask,
            d_in_mask=d_in_mask,
            d_out_mask=d_out_mask,
            d_in=self.tile_d_in,
            d_out=self.tile_d_out,
            d_in_pad=self.tile_d_in,
            T=self.tile_T,
        )[0]
        for idx, tile in enumerate(self.tiles):
            for seg in tile.segments:
                row_end = seg.tile_row_start + seg.row_len
                result[seg.name][seg.row_start:seg.row_start + seg.row_len, seg.col_start:seg.col_start + seg.col_len] = decoded[
                    idx,
                    seg.tile_row_start:row_end,
                    :seg.col_len,
                ]
        return result

    def tile_weight_and_context(
        self,
        tile: DecodeTile,
        base_state: dict[str, torch.Tensor],
        contexts: dict[str, torch.Tensor],
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        first_context = contexts[tile.segments[0].name]
        n = int(first_context.shape[0])
        W_tile = torch.zeros(self.tile_d_in, self.tile_d_out, device=device, dtype=torch.float32)
        X_tile = torch.zeros(n, self.tile_d_in, device=device, dtype=torch.float32)
        d_in_mask = torch.zeros(self.tile_d_in, device=device, dtype=torch.bool)
        d_out_mask = torch.zeros(self.tile_d_out, device=device, dtype=torch.bool)
        for seg in tile.segments:
            if seg.name not in contexts:
                raise KeyError(f'missing encoder context for {seg.name}')
            X_src = contexts[seg.name]
            if int(X_src.shape[0]) != n:
                X_src = X_src[:n]
            matrix = tensor_to_matrix(base_state[seg.name].to(device), self.specs[seg.name])
            row_end = seg.tile_row_start + seg.row_len
            W_tile[seg.tile_row_start:row_end, :seg.col_len] = matrix[
                seg.row_start:seg.row_start + seg.row_len,
                seg.col_start:seg.col_start + seg.col_len,
            ]
            X_tile[:, seg.tile_row_start:row_end] = X_src[:, seg.row_start:seg.row_start + seg.row_len]
            d_in_mask[seg.tile_row_start:row_end] = True
            d_out_mask[:seg.col_len] = True
        return W_tile, X_tile, d_in_mask, d_out_mask

    @torch.no_grad()
    def initialize_latents_from_encoder(self, base_state: dict[str, torch.Tensor], contexts: dict[str, torch.Tensor]) -> None:
        encoded: dict[str, torch.Tensor] = {}
        for idx, tile in enumerate(self.tiles):
            W_tile, X_tile, d_in_mask, d_out_mask = self.tile_weight_and_context(tile, base_state, contexts)
            W_b = W_tile.unsqueeze(0)
            X_b = X_tile.unsqueeze(0)
            d_in_mask_b = d_in_mask.unsqueeze(0)
            d_out_mask_b = d_out_mask.unsqueeze(0)
            T, d_in_pad, patch_mask, _structural_patch_mask, dist_var_by_patch, dist_patch_by_patch, dist_var_pooled = self.big_vae._encode_distribution_context(
                X_b,
                x_mask=None,
                d_in_mask=d_in_mask_b,
            )
            latents = self.big_vae._encode_latent_slots(
                W_b,
                T=T,
                d_in_pad=d_in_pad,
                patch_mask=patch_mask,
                d_out_mask=d_out_mask_b,
                dist_var_by_patch=dist_var_by_patch,
                dist_patch_by_patch=dist_patch_by_patch,
                dist_var_pooled=dist_var_pooled,
                return_debug_info=False,
            )
            encoded[tile.key] = latents.squeeze(0).detach().clone()
            if idx == 0 or (idx + 1) % 50 == 0 or idx + 1 == len(self.tiles):
                print(f'encoded latent tile {idx + 1}/{len(self.tiles)}', flush=True)
        for key, value in encoded.items():
            if tuple(self.latents[key].shape) != tuple(value.shape):
                raise RuntimeError(f'encoder latent shape mismatch for {key}: param={tuple(self.latents[key].shape)} encoded={tuple(value.shape)}')
            self.latents[key].copy_(value)

    def materialize_state(self, latent_override: dict[str, torch.Tensor] | None = None) -> dict[str, torch.Tensor]:
        state = {k: v.detach().clone() for k, v in self.direct_state.items()}
        matrices = self.decode_matrices(latent_override)
        for name, matrix in matrices.items():
            state[name] = matrix_to_tensor(matrix, self.specs[name])
        return state


def fit_latents_to_base(adapter: BigVAELatentAdapter, base_state: dict[str, torch.Tensor], steps: int, lr: float) -> None:
    opt = torch.optim.AdamW(adapter.latents.parameters(), lr=lr, weight_decay=0.0)
    targets = {name: tensor_to_matrix(base_state[name].to(device), spec) for name, spec in adapter.specs.items()}
    total = sum(target.numel() for target in targets.values())
    for step in range(1, int(steps) + 1):
        opt.zero_grad(set_to_none=True)
        decoded = adapter.decode_matrices()
        loss = sum(F.mse_loss(decoded[name], target, reduction='sum') for name, target in targets.items()) / max(1, total)
        loss.backward()
        opt.step()
        if step == 1 or step % max(1, int(steps) // 5) == 0 or step == int(steps):
            print(f'latent fit step={step}/{steps} mse={float(loss.detach().cpu()):.6e}')


def make_latent_direction(adapter: BigVAELatentAdapter, generator: torch.Generator) -> dict[str, torch.Tensor]:
    direction = {}
    for key, latent in adapter.latents.items():
        noise = torch.randn(latent.shape, generator=generator, dtype=latent.dtype, device='cpu').to(latent.device)
        direction[key] = noise / noise.norm().clamp_min(1e-12) * latent.detach().norm().clamp_min(1e-12)
    return direction


def make_latent_direction_pairs(adapter: BigVAELatentAdapter, num_pairs: int, seed: int):
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(seed))
    return [(make_latent_direction(adapter, generator), make_latent_direction(adapter, generator)) for _ in range(int(num_pairs))]


def call_tiny_vit_with_state(model: TinyStripeParityViT, state: dict[str, torch.Tensor], images: torch.Tensor) -> torch.Tensor:
    return functional_call(model, state, (images,))


@torch.no_grad()
def evaluate_latent_adapter(
    adapter: BigVAELatentAdapter,
    model_template: TinyStripeParityViT,
    loader: DataLoader,
    *,
    max_batches: int | None = None,
) -> tuple[float, float]:
    model_template.eval()
    state = adapter.materialize_state()
    total_loss = 0.0
    total_correct = 0
    total = 0
    for batch_idx, (images, labels) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = call_tiny_vit_with_state(model_template, state, images)
        loss = F.cross_entropy(logits, labels, reduction='sum')
        total_loss += float(loss.detach().cpu())
        total_correct += int((logits.argmax(dim=-1) == labels).sum().detach().cpu())
        total += int(labels.numel())
    return total_loss / max(1, total), total_correct / max(1, total)


def train_latents_on_task(
    adapter: BigVAELatentAdapter,
    model_template: TinyStripeParityViT,
    train_loader: DataLoader,
    val_loader: DataLoader,
    cfg: Config,
    *,
    max_eval_batches: int | None = None,
) -> None:
    steps = int(cfg.latent_task_train_steps)
    if steps <= 0:
        print('latent task training disabled: cfg.latent_task_train_steps <= 0')
        return
    model_template.eval()
    for param in model_template.parameters():
        param.requires_grad_(False)
    opt = torch.optim.AdamW(
        adapter.latents.parameters(),
        lr=float(cfg.latent_task_lr),
        weight_decay=float(cfg.latent_task_weight_decay),
    )
    batches = itertools.cycle(train_loader)
    log_every = max(1, int(cfg.latent_task_log_every))
    for step in range(1, steps + 1):
        images, labels = next(batches)
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        state = adapter.materialize_state()
        logits = call_tiny_vit_with_state(model_template, state, images)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        if float(cfg.latent_task_grad_clip_norm) > 0:
            torch.nn.utils.clip_grad_norm_(adapter.latents.parameters(), float(cfg.latent_task_grad_clip_norm))
        opt.step()
        if step == 1 or step % log_every == 0 or step == steps:
            batch_acc = float((logits.argmax(dim=-1) == labels).float().mean().detach().cpu())
            val_loss, val_acc = evaluate_latent_adapter(adapter, model_template, val_loader, max_batches=max_eval_batches)
            print(
                f'latent task step={step}/{steps} loss={float(loss.detach().cpu()):.4f} '
                f'batch_acc={batch_acc:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
            )


In [ ]:
latent_results = []
latent_metrics = pd.DataFrame()

if not BIG_VAE_CHECKPOINT:
    print('BIG_VAE_CHECKPOINT is empty; skipping latent-induced landscape.')
else:
    big_vae = load_frozen_big_vae(BIG_VAE_CHECKPOINT, device)
    adapter = BigVAELatentAdapter(big_vae, base_state, cfg).to(device)
    print('latent tiles:', len(adapter.tiles))
    print('decoded linear params:', adapter.decoded_numel())
    print('latent params:', adapter.latent_numel())
    print('effective decoded/latent:', adapter.decoded_numel() / max(1, adapter.latent_numel()))

    mode = str(cfg.latent_base_mode).strip().lower()
    if mode not in {'encoder', 'fit', 'encoder_fit', 'random'}:
        raise ValueError(f'unsupported latent_base_mode={cfg.latent_base_mode!r}')
    if mode in {'encoder', 'encoder_fit'}:
        print('initializing latent base point from BigVAE encoder')
        adapter.initialize_latents_from_encoder(base_state, encoder_contexts)
    if mode in {'fit', 'encoder_fit'} and int(cfg.latent_fit_steps) > 0:
        print('refining latent base point by decoded-weight MSE')
        fit_latents_to_base(adapter, base_state, cfg.latent_fit_steps, cfg.latent_fit_lr)

    latent_template_model = TinyStripeParityViT(cfg).to(device)
    latent_template_model.load_state_dict(base_state, strict=True)
    latent_pre_task_loss, latent_pre_task_acc = evaluate_latent_adapter(
        adapter,
        latent_template_model,
        landscape_loader,
        max_batches=landscape_max_batches,
    )
    print(f'latent_pre_task_{landscape_split}_loss', latent_pre_task_loss, f'latent_pre_task_{landscape_split}_acc', latent_pre_task_acc)
    print('training latent slots on stripe-parity task loss')
    train_latents_on_task(
        adapter,
        latent_template_model,
        train_loader,
        val_loader,
        cfg,
        max_eval_batches=landscape_max_batches,
    )

    latent_base_state = adapter.materialize_state()
    latent_model = TinyStripeParityViT(cfg).to(device)
    latent_model.load_state_dict(latent_base_state, strict=True)
    latent_base_loss, latent_base_acc = evaluate_loss(latent_model, landscape_loader, max_batches=landscape_max_batches)
    print(f'latent_base_{landscape_split}_loss', latent_base_loss, f'latent_base_{landscape_split}_acc', latent_base_acc)
    latent_pairs = make_latent_direction_pairs(adapter, cfg.num_directions, cfg.seed + 2000)

    def latent_eval_fn(pair, alpha: float, beta: float) -> float:
        d1, d2 = pair
        override = {
            key: adapter.latents[key] + float(alpha) * d1[key] + float(beta) * d2[key]
            for key in adapter.latents.keys()
        }
        state = adapter.materialize_state(override)
        latent_model.load_state_dict(state, strict=True)
        loss, _acc = evaluate_loss(latent_model, landscape_loader, max_batches=landscape_max_batches)
        return loss

    latent_results = evaluate_landscape(
        kind='latent',
        direction_pairs=latent_pairs,
        base_loss=latent_base_loss,
        eval_fn=latent_eval_fn,
        grid_points=cfg.grid_points,
        rho_max=cfg.rho_max,
    )
    latent_metrics = compute_landscape_metrics(latent_results, cfg.rho_values, cfg.tau_values)
    display(latent_metrics)
    plot_landscape(
        latent_results[0],
        'BigVAE latent-induced stripe-parity landscape: direction 0',
        save_path=artifact_dir / 'latent_direction_0_2d.png',
    )
    plot_landscape_3d(
        latent_results[0],
        'Interactive 3D BigVAE latent-induced landscape: direction 0',
        save_html_path=artifact_dir / 'latent_direction_0_3d.html',
    )


In [ ]:
all_metrics = pd.concat([raw_metrics, latent_metrics], ignore_index=True) if len(latent_metrics) else raw_metrics.copy()
display(all_metrics)

plot_metric_boxplots(all_metrics, 'S_rho', save_path=artifact_dir / 'metric_S_rho_boxplot.png')
plot_metric_boxplots(all_metrics, 'M_rho', save_path=artifact_dir / 'metric_M_rho_boxplot.png')
for tau in cfg.tau_values:
    col = f'A_frac_tau={tau:g}_rho'
    if col in all_metrics.columns:
        plot_metric_boxplots(all_metrics, col, save_path=artifact_dir / f'metric_{col.replace("=", "_").replace(".", "p")}_boxplot.png')

out_dir = artifact_dir
out_dir.mkdir(parents=True, exist_ok=True)
all_metrics.to_csv(out_dir / 'landscape_metrics.csv', index=False)

save_payload: dict[str, Any] = {}
for result in raw_results:
    save_payload[f'raw_losses_direction_{result.direction_idx}'] = result.losses
for result in latent_results:
    save_payload[f'latent_losses_direction_{result.direction_idx}'] = result.losses
if raw_results:
    save_payload['raw_alphas'] = raw_results[0].alphas
    save_payload['raw_betas'] = raw_results[0].betas
    save_payload['raw_base_loss'] = np.array([raw_results[0].base_loss], dtype=np.float64)
if latent_results:
    save_payload['latent_alphas'] = latent_results[0].alphas
    save_payload['latent_betas'] = latent_results[0].betas
    save_payload['latent_base_loss'] = np.array([latent_results[0].base_loss], dtype=np.float64)
np.savez_compressed(out_dir / 'landscape_results.npz', **save_payload)
print('saved metrics to', out_dir / 'landscape_metrics.csv')
print('saved raw/latent grids to', out_dir / 'landscape_results.npz')
